# 7.25 Optical flow

Optical flow asks every pixel a delicate question: where did this same brightness move between two nearby frames?

Optical flow turns two frames into a field of displacement vectors. Gradients explain the local brightness-constancy equation, while warping checks whether a proposed motion reconstructs the next frame.

Save a copy to Drive to edit.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

rng = np.random.default_rng(7)
np.set_printoptions(precision=3, suppress=True)

## The concept, built once

The lesson formula is

$$I_xu+I_yv+I_t=0.$$

We verify the $3	imes3	imes2=18$ scalar field, a zero residual, aperture ambiguity, the $(1,1)$ solution, and a zero photometric warp error.

In [ ]:
def flow_residual(ix, iy, it, u, v):
    return ix * u + iy * v + it


def solve_tiny_flow(equations):
    a = np.array([[row[0], row[1]] for row in equations], dtype=float)
    b = np.array([-row[2] for row in equations], dtype=float)
    return np.linalg.solve(a, b)


def shift_image(img, dx, dy, fill=0.0):
    out = np.full_like(img, fill)
    h, w = img.shape
    src_y0 = max(0, -dy)
    src_y1 = min(h, h - dy)
    src_x0 = max(0, -dx)
    src_x1 = min(w, w - dx)
    dst_y0 = max(0, dy)
    dst_y1 = min(h, h + dy)
    dst_x0 = max(0, dx)
    dst_x1 = min(w, w + dx)
    out[dst_y0:dst_y1, dst_x0:dst_x1] = img[src_y0:src_y1, src_x0:src_x1]
    return out


flow = np.zeros((3, 3, 2), dtype=float)
flow[:, :, 0] = 1
assert flow.size == 18
assert tuple(flow[1, 1]) == (1.0, 0.0)
assert flow_residual(2, 0, -2, 1, 0) == 0
assert flow_residual(2, 0, -2, 1, 5) == 0
uv = solve_tiny_flow([(2, 0, -2), (0, 3, -3)])
assert tuple(uv.astype(int)) == (1, 1)
frame = np.arange(9).reshape(3, 3)
assert frame[1, 2] - frame[1, 1] == 1
print("scalar count", flow.size)
print("center vector", tuple(flow[1, 1]))
print("brightness residual", flow_residual(2, 0, -2, 1, 0))
print("aperture residual", flow_residual(2, 0, -2, 1, 5))
print("two-equation solution", uv)

## Dataset ladder

For flow we synthesize frame pairs with increasing displacement, distractors, exposure change, and occlusion. The metric is endpoint error, EPE.

In [ ]:
def make_blob(size, center, radius):
    yy, xx = np.mgrid[0:size, 0:size]
    cy, cx = center
    blob = np.exp(-((xx - cx) ** 2 + (yy - cy) ** 2) / (2 * radius ** 2))
    return blob


def make_flow_ladder():
    specs = [
        ("D1 one-pixel right", 24, 1, 0, 0.00, 0.0),
        ("D2 diagonal", 24, 1, 1, 0.01, 0.0),
        ("D3 larger motion", 28, 2, 1, 0.02, 0.0),
        ("D4 noisy motion", 28, 3, 1, 0.04, 0.0),
        ("D5 exposure occlusion", 32, 3, 2, 0.04, 0.35),
    ]
    rungs = []
    for seed, spec in enumerate(specs):
        name, size, dx, dy, noise, occlusion = spec
        local_rng = np.random.default_rng(seed + 20)
        frame1 = make_blob(size, (size // 2, size // 2), 3.0)
        frame1 = frame1 + 0.4 * make_blob(size, (size // 3, size // 3), 2.0)
        frame2 = shift_image(frame1, dx, dy)
        frame2 = frame2 + local_rng.normal(0.0, noise, size=frame2.shape)
        if occlusion > 0:
            frame2[4:12, 4:12] = occlusion
        frame1 = np.clip(frame1, 0.0, 1.0)
        frame2 = np.clip(frame2, 0.0, 1.0)
        rungs.append({"name": name, "frame1": frame1, "frame2": frame2, "true": np.array([dx, dy])})
    return rungs


def estimate_flow(frame1, frame2, search=4, robust=False):
    best_error = np.inf
    best = np.array([0, 0])
    mask = np.ones_like(frame1, dtype=bool)
    if robust:
        mask = frame1 > np.percentile(frame1, 45)
    for dy in range(-search, search + 1):
        for dx in range(-search, search + 1):
            shifted = shift_image(frame1, dx, dy)
            error = np.mean((shifted[mask] - frame2[mask]) ** 2)
            if error < best_error:
                best_error = error
                best = np.array([dx, dy])
    return best, best_error


flow_rungs = make_flow_ladder()
for rung in flow_rungs:
    print(rung["name"], "true", tuple(rung["true"]), "shape", rung["frame1"].shape)

fig, axes = plt.subplots(1, 5, figsize=(12, 2.4))
for ax, rung in zip(axes, flow_rungs):
    ax.imshow(rung["frame1"], cmap="gray", vmin=0, vmax=1)
    ax.set_title(rung["name"])
    ax.axis("off")
plt.tight_layout()

In [ ]:
flow_results = []
for rung in flow_rungs:
    pred, error = estimate_flow(rung["frame1"], rung["frame2"])
    epe = float(np.linalg.norm(pred - rung["true"]))
    flow_results.append({"name": rung["name"], "pred": pred, "epe": epe, "error": error})

print("rung                 pred      true      EPE")
for rung, result in zip(flow_rungs, flow_results):
    print(f"{rung['name']:<20} {tuple(result['pred'])!s:<9} {tuple(rung['true'])!s:<9} {result['epe']:.2f}")

In [ ]:
fig, axes = plt.subplots(2, 5, figsize=(13, 5))
for i, rung in enumerate(flow_rungs):
    axes[0, i].imshow(rung["frame2"], cmap="gray", vmin=0, vmax=1)
    pred = flow_results[i]["pred"]
    axes[0, i].arrow(6, 6, pred[0], pred[1], color="red", width=0.25)
    axes[0, i].set_title(rung["name"])
    axes[0, i].axis("off")

scores = [item["epe"] for item in flow_results]
axes[1, 0].plot(range(1, 6), scores, marker="o")
axes[1, 0].set_xlabel("rung")
axes[1, 0].set_ylabel("EPE")
axes[1, 0].set_title("endpoint error")
for ax in axes[1, 1:]:
    ax.axis("off")
plt.tight_layout()

## Pitfall on D5: brightness constancy and occlusion can fail

Exposure changes and occluders can make a photometric search prefer the wrong displacement. A robust mask improves the comparison by ignoring low-evidence pixels.

In [ ]:
def estimate_flow_masked(frame1, frame2, search=4):
    base_mask = frame1 > np.percentile(frame1, 45)
    target_mask = frame2 < 0.95
    mask = base_mask & target_mask
    target = frame2[mask]
    target = (target - target.mean()) / (target.std() + 1e-6)
    best_error = np.inf
    best = np.array([0, 0])
    for dy in range(-search, search + 1):
        for dx in range(-search, search + 1):
            shifted = shift_image(frame1, dx, dy)
            sample = shifted[mask]
            sample = (sample - sample.mean()) / (sample.std() + 1e-6)
            error = np.mean((sample - target) ** 2)
            if error < best_error:
                best_error = error
                best = np.array([dx, dy])
    return best, best_error


hard = flow_rungs[-1]
exposed = hard["frame2"] * 0.3 + 0.4
exposed[0:12, 20:32] = 1.0
wrong, wrong_error = estimate_flow(hard["frame1"], exposed, robust=False)
fixed, fixed_error = estimate_flow_masked(hard["frame1"], exposed)
wrong_epe = np.linalg.norm(wrong - hard["true"])
fixed_epe = np.linalg.norm(fixed - hard["true"])
print("plain", tuple(wrong), "EPE", round(float(wrong_epe), 2), "photometric", round(float(wrong_error), 4))
print("masked", tuple(fixed), "EPE", round(float(fixed_epe), 2), "normalized", round(float(fixed_error), 4))

## Evaluate it + Practice

- Metric: endpoint error $\|\hat u-u\|_2$; no-skill baseline is zero flow.
- Sanity check: D1 should recover one-pixel right motion.
- Ablation: remove the mask on D5 and occlusion hurts.
- Failure signal: low photometric error on occluded pixels can still be wrong motion.

Practice:
1. Increase the search range.
2. Add an exposure normalization step.
3. Try a second moving blob.